In [8]:
from datetime import datetime, timedelta
from pathlib import Path
import pandas as pd
import dwcalendar_2

In [9]:
def next_week_window(today=None):
    today = today or datetime.now().date()
    days_ahead = (0 - today.weekday() + 7) % 7
    if days_ahead == 0:
        days_ahead = 7
    start = today + timedelta(days=days_ahead)
    end = start + timedelta(days=6)
    return start, end

def build_note(date_obj):
    return f"AUTOHOL: HOLIDAY CHECK — {date_obj.strftime('%Y-%m-%d')}"

In [10]:
def main(excel_path: str, *, dry_run: bool = False):
    start, end = next_week_window()
    print(f"Checking holidays between {start} and {end}")

    # 1) Read Excel
    df = pd.read_excel(excel_path)

    # 2) Drop rows whose "Holiday Observance" contains "regional" (case-insensitive)
    df_copy = df.loc[~df["Holiday Observance"].str.contains("regional", case=False, na=False)].copy()

    # 3) Clean column types/whitespace on the filtered copy
    df_copy["Holiday Date"] = pd.to_datetime(df_copy["Holiday Date"]).dt.date
    df_copy["Country Name"] = df_copy["Country Name"].astype(str).str.strip()
    df_copy["Holiday Name"] = df_copy["Holiday Name"].astype(str).str.strip()

    # 4) Filter to the next-week window
    mask = (df_copy["Holiday Date"] >= start) & (df_copy["Holiday Date"] <= end)
    week = df_copy.loc[mask, ["Country Name", "Holiday Date", "Holiday Name"]].copy()

    if week.empty:
        print("No non-regional holidays next week.")
        return
    
    # --- 5) Connect to autocalendar DB ---
    driver = dwcalendar_2.get_odbc_driver()
    dwcalendar_2.connect_to_db(
        f'DRIVER={{{driver}}};SERVER=10.1.4.6;PORT=3306;DATABASE=autocalendar;UID=autocal;PWD=AutoCal_Pwd'
    )

    week = week.sort_values(["Country Name", "Holiday Date", "Holiday Name"])
    successes, misses = 0, 0

    try:
        # --- 6) Loop and post notes ---
        for _, r in week.iterrows():
            country = r["Country Name"]
            hdate = r["Holiday Date"]
            hname = r["Holiday Name"]

            record = dwcalendar_2.find_record(country, hdate)
            if record is None:
                print(f"[MISS] No autocalendar record for {country} on {hdate}")
                misses += 1
                continue

            note = build_note(hdate)
            if dry_run:
                print(f"[DRY-RUN] Would post: {country}: {hname} ({hdate}) → {note}")
            else:
                final_note = dwcalendar_2.write_note(record, note)
                print(f"[OK] {country}: {hname} ({hdate}) → note posted: {final_note}")
            successes += 1
    finally:
        dwcalendar_2.close()

    print(f"\nDone. Notes posted: {successes}. Misses: {misses}")


In [ ]:
if __name__ == "__main__":
    EXCEL_PATH = r"Q++ Worldwide Public Holidays ISO-2025.XLS"
    main(EXCEL_PATH,dry_run=False)